In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pylab as plt

import seaborn as sns

from skspatial.objects import Line, Plane
from skspatial.plotting import plot_3d


from skspatial.objects import Line, Cylinder, Point, Points
from skspatial.plotting import plot_3d

import phasespace

import tensorflow

import bisect
import numpy as np
import matplotlib.pylab as plt
import pandas as pd

import seaborn as sns

import numpy as np
from scipy.stats import multivariate_normal

import numpy as np
from scipy.interpolate import griddata
from scipy.integrate import quad, trapezoid
from scipy.interpolate import CubicSpline

import matplotlib.pylab as plt
from scipy import stats
from matplotlib import cm
from matplotlib.ticker import LinearLocator

from scipy.interpolate import LinearNDInterpolator

import eloss_tools


import dm_generation_tools as dgt
import detector_simulation_tools as dst

import glob

import time

####################################
import warnings
# Suppress all warnings
warnings.filterwarnings("ignore")


import pickle

In [ ]:
print(f'{np.__version__ = }')

print(f'{phasespace.__version__ = }')
print(f'{tensorflow.__version__ = }')

In [ ]:
######################################################################

# Masses in GeV/c^2
MASSES_A = [.220]
MASSES_DM = [1000, 7000]

######################################################################
# Set the depths
######################################################################
# Depths are in meters. 0 depth is *center* of detector
# so -7.5 is just at the bottom of CMS
#
# Values should be negative
#
# When the depth values are a list, then it generates the muon origins uniformly
# over a volume 

# Plane
depths = [-17.5, -1000]

# Volume
#depths = [[-100,-200]]

# When the radius is None, it calculates the radius based on an angle of 91 degrees 
radii = [40, None]

# Dark matter model
# "floating" - unform direction for dark photon direction, but constrained to go "up"
#dm_model = 'floating'

# "core" - dark photons are coming straight up (all momentum in y)
#dm_model = 'core'

# "momentum_constrained" unphysical model with dark photon at rest, but allows
# us to fix the magnitude of the momentum of the muon
# Uniform in direction
dm_model = 'momentum_constrained'

# Output directory for intermediate files and summary file
output_directory = 'OUTPUT_FILES'

# Do you want to save all the events or not? 
# If you set this to be true, then try it with a small number of events and only 1 "trial" file
#SAVE_ONLY_DETECTED = False
SAVE_ONLY_DETECTED = True

##################################################
# Size of the detector
#
# Note that we have a text string (tracker_tag) that
# gets added to file names.
##################################################
# Hits the tracker
#detector_radius = 1.5
#half_len = 1
#tracker_tag = 'TRACKER_VOL_'

# Full CMS
detector_radius = 7.5
half_len = 10
tracker_tag = ''

######################################################################
# Generate!
######################################################################

for radius in radii:
    for depth in depths:
        print(f'radius: {radius}     depth: {depth}')
        
        save_to_file = True
        
        #nevents = 1_000_000
        # For testing
        nevents = 1000

        # Trials is not the right word, but these are the number of intermediate files that get produced
        # and then combined into one file.
        #
        # This is useful when you only want to save events that hit the detector
        #ntrials = 1000
        ntrials = 5

        # You might want to have more trials if you are very deep and have a low acceptance 
        '''
        if depth < -200 and depth > -999:
            ntrials = 2000
        elif depth <= -999:
            ntrials = 50000
        '''
        
        ##############################################################        
        # Generate all the files!!!!!!
        for i in range(0,ntrials):
            print(f'{i} --------------------------------------------------------')
            print(f'Generate {nevents} events...')
            df_decays,tag = dgt.generate_many_events(MASSES_A=MASSES_A, MASSES_DM=MASSES_DM, nevents_to_generate=nevents, \
                                         radius=radius, depth=depth, dm_model=dm_model, \
                                                 detector_radius=detector_radius, half_len=half_len, \
                                         SAVE_ONLY_DETECTED=SAVE_ONLY_DETECTED, eloss_dict_name='eloss_dictionary_11032025_v2.pkl', 
                                            save_to_file=save_to_file, additional_tag=f'_{tracker_tag}{i:06d}', \
                                                output_directory=output_directory)
        ##############################################################

        
        ##############################################################
        # Merge the "trial" files
        ##############################################################
        partial_tag = tag[0:-6]
        print(f'partial_tag: {partial_tag}')
        
        files = glob.glob(f'{output_directory}/generated_data_{partial_tag}[0-9]*.parquet')

        #print('files')
        #print(files)
        
        # Merge them
        dfs = []
        for i,filename in enumerate(files):
            #print(filename)
            dfs.append( pd.read_parquet(filename))
            
        print(f'\n Merging {len(dfs)} intermediate files...\n')
        df_decays = pd.concat(dfs)
        df_decays.to_parquet(f'{output_directory}/generated_data_{partial_tag}COMBINED.parquet')

        # Delete the intermediate dataframes to free up memory
        del df_decays
        del dfs[:]


In [ ]:
# The files of interest have "COMBINED" in the name

!ls -ltr OUTPUT_FILES/*COMBINED.parquet